## PDF to Chroma Pipeline

This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma, and runs a retrieval query.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. Prepare Paths and Configuration

In [3]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores')

In [4]:
env_path = project_root / ".env"
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

PDF path: /Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
Collection name: rag-pipeline
Persist directory: /Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/notebooks/chroma_langchain_db


In [5]:
load_dotenv(dotenv_path=env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {env_path}")

Loaded environment from: /Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/.env


In [6]:
# Reuse the same embedding model as the other notebooks in this repo.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Embedding model is ready.")

Embedding model is ready.


## 2. Add Small Display Helpers

In [7]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Load the PDF

In [9]:
loader = PyPDFLoader(str(pdf_path))

In [10]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 6


In [11]:
docs[0].page_content

'Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo\nFor a long time, the most common experience with AI felt theatrical. You typed a question, the model\nanswered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a\nfollow-up that required memory, factual grounding, or a small sequence of actions, and the system often fell\napart. It could sound confident without being connected to anything real. That gap between fluency and\nusefulness is exactly where AI agents enter the picture.\nAn AI agent is interesting not because it sounds human, but because it behaves like software with intent. It\ncan take a goal, figure out what it needs in order to make progress, and work through a sequence of steps\ninstead of improvising a single reply. In the simplest form, that might mean

In [12]:
print(f"First page preview: {preview_text(docs[0].page_content)}")
print(f"First page metadata: {docs[0].metadata}")

First page preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First page metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': '/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 4. Split the PDF into Chunks

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [14]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


In [15]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

First chunk preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': '/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 5. Store Chunks in Chroma

In [16]:
collection_name

'rag-pipeline'

In [18]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


## 6. Retrieve Relevant Chunks

In [19]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [20]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=2 | source=/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
3. page=2 | source=/Use

In [29]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page no: {doc.metadata.get('page_label')}")
    # print(f"page_no. {doc.metadata.get("page_label")}")

Score: 0.8492
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page no: 3
Score: 0.8492
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page no: 3
